#Rhitmic Trading Journal and Account Tracker


In [ ]:
#Notes 



#notes
Still needs: 
i.  color coordination triggered by amount 
ii. Tkinter or HTML interface generated >>> convert to Windows EXE file, Apple format? Android? 



In [21]:
import os
import pandas as pd
import glob
from datetime import datetime
import openpyxl
from openpyxl.styles import Font, Alignment

# Define the paths to the folders
dashboard_path = r"C:\Users\Wolfrank\Desktop\TraderTracker\Rithmic_Dashboard"
performance_path = r"C:\Users\Wolfrank\Desktop\TraderTracker\Rithmic_Performance"
output_path = r"C:\Users\Wolfrank\Desktop\TraderTracker\TraderTrackrr.xlsx"

# List of known accounts (will be supplemented with any found in data)
known_accounts = [
    "PA-APEX-1708-60", "PA-APEX-1708-61", "PA-APEX-1708-68", "PA-APEX-1708-69",
    "PA-APEX-1708-73", "PA-APEX-1708-74", "PA-APEX-1708-75", "PA-APEX-1708-76",
    "PA-APEX-1708-77", "PA-APEX-1708-79", "PA-APEX-1708-80", "PA-APEX-1708-81",
    "PA-APEX-1708-83"
]

def find_csv_files(folder_path):
    """Find all CSV files in a folder and its subfolders"""
    return glob.glob(os.path.join(folder_path, "**/*.csv"), recursive=True)

def extract_date_from_filename(filename):
    """Extract date from filename, assuming format contains a date string"""
    # This function may need adjustment based on your actual filename format
    try:
        # Try to extract date from filename (adjust this pattern based on your filenames)
        base_name = os.path.basename(filename)
        # Example: if filename is like "data_2023-05-25.csv" or contains date in some format
        date_parts = [part for part in base_name.split('_') if '-' in part]
        if date_parts:
            date_str = date_parts[0].split('.')[0]
            return pd.to_datetime(date_str).strftime('%Y-%m-%d')
    except:
        pass
    
    # If date can't be extracted from filename, use file modification date
    file_mtime = os.path.getmtime(filename)
    return datetime.fromtimestamp(file_mtime).strftime('%Y-%m-%d')

def process_dashboard_files(file_list):
    """Process dashboard CSV files and merge them by date and account"""
    all_data = []
    
    for file_path in file_list:
        date = extract_date_from_filename(file_path)
        
        try:
            # Read dashboard data
            df = pd.read_csv(file_path)
            
            # Add date column if it doesn't exist
            if 'Date' not in df.columns:
                df['Date'] = date
                
            all_data.append(df)
        except Exception as e:
            print(f"Error processing dashboard file {file_path}: {e}")
    
    # Combine all dataframes
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

def process_performance_files(file_list):
    """Process performance CSV files and merge them by date and account"""
    all_data = []
    
    for file_path in file_list:
        date = extract_date_from_filename(file_path)
        
        try:
            # Read performance data
            df = pd.read_csv(file_path)
            
            # Add date column if it doesn't exist
            if 'Date' not in df.columns:
                df['Date'] = date
                
            all_data.append(df)
        except Exception as e:
            print(f"Error processing performance file {file_path}: {e}")
    
    # Combine all dataframes
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

def merge_data(dashboard_df, performance_df):
    """Merge dashboard and performance data by Date and Account"""
    # Ensure both dataframes have Date and Account columns
    if dashboard_df.empty or performance_df.empty:
        print("Warning: One or both dataframes are empty!")
        
    # Merge the dataframes
    if not dashboard_df.empty and not performance_df.empty:
        # Standardize column names if needed
        dashboard_df.columns = [col.strip() for col in dashboard_df.columns]
        performance_df.columns = [col.strip() for col in performance_df.columns]
        
        # Merge on Date and Account
        merged_df = pd.merge(
            dashboard_df, 
            performance_df, 
            on=['Date', 'Account'], 
            how='outer',
            suffixes=('', '_perf')
        )
        
        return merged_df
    elif not dashboard_df.empty:
        return dashboard_df
    elif not performance_df.empty:
        return performance_df
    else:
        return pd.DataFrame()

def calculate_additional_metrics(merged_df):
    """Calculate additional metrics needed for the output spreadsheet"""
    df = merged_df.copy()
    
    # These column calculations will depend on your exact data structure
    # Below are placeholders - adjust according to your actual column names and data
    
    # Calculate average profit for wins and losses if possible
    if 'Trade P&L' in df.columns and 'Winning Trades' in df.columns and 'Losing Trades' in df.columns:
        df['Avg Profit Wins'] = df.apply(
            lambda row: row['Trade P&L'] / row['Winning Trades'] if row['Winning Trades'] > 0 else 0, 
            axis=1
        )
        df['Avg Profit Loss'] = df.apply(
            lambda row: row['Trade P&L'] / row['Losing Trades'] if row['Losing Trades'] < 0 else 0, 
            axis=1
        )
    
    # Add placeholders for columns we don't have data for yet
    if 'Shorts' not in df.columns:
        df['Shorts'] = None
    if 'Longs' not in df.columns:
        df['Longs'] = None
    if 'Avg Time Held' not in df.columns:
        df['Avg Time Held'] = None
    
    # Rename columns for consistency with requested output
    column_mapping = {
        'P&L': 'PnL',
        'Winning Trades': 'Win Count',
        'Losing Trades': 'Loss Count',
        'Trade Count': 'Total Trades',
        'Auto Liquidate Threshold Value': 'Auto Liquidate Value'
    }
    
    df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
    
    return df

def create_summary_sheet(merged_data, accounts_list):
    """Create summary sheet with account statistics"""
    pa_accounts_data = []
    evaluation_accounts_data = []
    
    for account in accounts_list:
        account_data = merged_data[merged_data['Account'] == account]
        
        if not account_data.empty:
            # Calculate metrics for summary
            days_traded = account_data['Date'].nunique()
            
            # Get latest balance if available
            balance = account_data['Account Balance'].iloc[-1] if 'Account Balance' in account_data.columns else None
            
            # Get latest auto liquidate value if available
            auto_liquidate_value = None
            if 'Auto Liquidate Value' in account_data.columns:
                auto_liquidate_value = account_data['Auto Liquidate Value'].iloc[-1]
            
            # Calculate distance to auto liquidate
            distance_to_auto_liquidate = None
            if balance is not None and auto_liquidate_value is not None:
                distance_to_auto_liquidate = balance - auto_liquidate_value
            
            # Find largest daily profit
            if 'PnL' in account_data.columns:
                largest_profit = account_data.groupby('Date')['PnL'].sum().max()
            else:
                largest_profit = None
            
            # Set Goal Balance based on account number
            goal_balance = 55600  # Default for most accounts
            if account == "PA-APEX-1708-77":
                goal_balance = 311000
            elif account == "PA-APEX-1708-79":
                goal_balance = 158000
            
            # Set default Threshold for Payout value based on account number
            threshold_for_payout = 52600  # Default for most accounts
            if account == "PA-APEX-1708-77":
                threshold_for_payout = 307600
            elif account == "PA-APEX-1708-79":
                threshold_for_payout = 155750
                
            # Calculate Distance to 500 Payout
            distance_to_500_payout = None
            if balance is not None:
                distance_to_500_payout = threshold_for_payout - balance
            
            # Set Goal Payout threshold based on account number
            goal_payout_threshold = 55600  # Default for most accounts
            if account == "PA-APEX-1708-77":
                goal_payout_threshold = 311000
            elif account == "PA-APEX-1708-79":
                goal_payout_threshold = 158000
                
            # Set Daily Profit Limit for 500 based on account number (30% of threshold differences)
            daily_profit_limit_500 = 0.3 * 2600  # Default for most accounts (30% of 2600)
            if account == "PA-APEX-1708-77":
                daily_profit_limit_500 = 0.3 * 7600  # 30% of 7600
            elif account == "PA-APEX-1708-79":
                daily_profit_limit_500 = 0.3 * 5100  # 30% of 5100
                
            # Set Daily Profit Limit for Goal based on account number (30% of goal differences)
            daily_profit_limit_goal = 0.3 * 5600  # Default for most accounts (30% of 5600)
            if account == "PA-APEX-1708-77":
                daily_profit_limit_goal = 0.3 * 11000  # 30% of 11000
            elif account == "PA-APEX-1708-79":
                daily_profit_limit_goal = 0.3 * 8000  # 30% of 8000
                
            # Calculate Distance to Goal Payout
            distance_to_goal_payout = None
            if balance is not None:
                distance_to_goal_payout = goal_payout_threshold - balance
                
            account_summary = {
                'Account': account,
                'Balance': balance,
                'Goal Balance': goal_balance,
                'Threshold for Payout': threshold_for_payout,
                'Days Traded': days_traded,
                'Largest Profit for Account in a Day': largest_profit,
                'Distance to Auto Liquidate': distance_to_auto_liquidate,
                'Notes on Account': '',  # Empty for user to fill in
                'Withdrawal Notes': '',  # Empty for user to fill in
                'Daily Profit Limit for 500': daily_profit_limit_500,
                'Daily Profit Limit for Goal': daily_profit_limit_goal,
                'Distance to 500 Payout': distance_to_500_payout,
                'Distance to Goal Payout': distance_to_goal_payout
            }
            
            # Separate PA-APEX accounts from others (evaluations)
            if account.startswith('PA-APEX'):
                pa_accounts_data.append(account_summary)
            else:
                # For non-PA-APEX accounts, mark them as Evaluations
                account_summary['Notes on Account'] = 'Evaluation'
                evaluation_accounts_data.append(account_summary)
    
    # Create the combined dataframe with a gap between PA accounts and evaluations
    if evaluation_accounts_data:
        # Create empty rows for the gap
        empty_rows = [pd.DataFrame([{col: None for col in pa_accounts_data[0].keys()}]) for _ in range(4)]
        
        # Combine PA accounts, empty rows, and evaluation accounts
        all_dfs = [pd.DataFrame(pa_accounts_data)]
        all_dfs.extend(empty_rows)
        all_dfs.append(pd.DataFrame(evaluation_accounts_data))
        
        return pd.concat(all_dfs, ignore_index=True)
    else:
        # If no evaluation accounts, just return PA accounts
        return pd.DataFrame(pa_accounts_data)

def main():
    print("Starting Trader Tracking Data Merger...")
    
    # Find all CSV files in both folders
    dashboard_files = find_csv_files(dashboard_path)
    performance_files = find_csv_files(performance_path)
    
    print(f"Found {len(dashboard_files)} dashboard files and {len(performance_files)} performance files")
    
    # Process files from both folders
    dashboard_data = process_dashboard_files(dashboard_files)
    performance_data = process_performance_files(performance_files)
    
    # Merge the data
    merged_data = merge_data(dashboard_data, performance_data)
    
    # Discover all unique accounts from the data
    accounts = []
    if not merged_data.empty and 'Account' in merged_data.columns:
        discovered_accounts = merged_data['Account'].unique().tolist()
        # Combine with known accounts and remove duplicates
        accounts = list(set(known_accounts + discovered_accounts))
        print(f"Found {len(accounts)} unique accounts in the data")
    else:
        accounts = known_accounts
        print("Using predefined account list as no accounts were discovered in data")
        
    # Calculate additional metrics
    enhanced_data = calculate_additional_metrics(merged_data)
    
    # Create summary sheet data
    summary_data = create_summary_sheet(enhanced_data, accounts)
    
    # Process data for all accounts (for consolidated view)
    all_accounts_data = pd.DataFrame()
    
    # Prepare all account data first
    for account in accounts:
        account_data = enhanced_data[enhanced_data['Account'] == account]
        
        if not account_data.empty:
            # Select and order columns for individual account sheets
            columns_to_include = [
                'Date', 'Account', 'PnL', 'Total Commission', 'Account Balance',
                'Auto Liquidate Value', 'Win Count', 'Loss Count', 'Total Trades',
                'Avg Profit Wins', 'Avg Profit Loss', 'Shorts', 'Longs', 'Avg Time Held'
            ]
            
            # Filter columns that actually exist in our data
            actual_columns = [col for col in columns_to_include if col in account_data.columns]
            
            # Create missing columns as empty
            for col in columns_to_include:
                if col not in account_data.columns:
                    account_data[col] = None
            
            # Sort by date
            account_data = account_data.sort_values('Date')
            
            # Add trading day count column (only counting days with trades)
            account_data['Trading Day Count'] = None
            day_count = 0
            
            for idx, row in account_data.iterrows():
                # Increment counter only if there were trades that day
                if pd.notnull(row['Total Trades']) and row['Total Trades'] > 0:
                    day_count += 1
                account_data.at[idx, 'Trading Day Count'] = day_count
            
            # Add Trading Day Count to the columns list
            columns_to_include.append('Trading Day Count')
            
            # Add this account's data to the consolidated dataframe
            all_accounts_data = pd.concat([all_accounts_data, account_data[columns_to_include]])
    
    # Create Excel writer
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # First, write the summary sheet
        summary_data.to_excel(writer, sheet_name='TraderTrackrr', index=False)
        
        # Second, write consolidated account data sheet
        if not all_accounts_data.empty:
            # Sort the consolidated data by Account and Date
            all_accounts_data = all_accounts_data.sort_values(['Account', 'Date'])
            all_accounts_data.to_excel(writer, sheet_name='AccountData', index=False)
        
        # Finally, write individual account sheets
        for account in accounts:
            account_data = all_accounts_data[all_accounts_data['Account'] == account]
            
            if not account_data.empty:
                # Write the account data to its own sheet
                account_data.to_excel(
                    writer, 
                    sheet_name=account, 
                    index=False
                )
        
        # Write consolidated account data sheet
        if not all_accounts_data.empty:
            # Sort the consolidated data by Account and Date
            all_accounts_data = all_accounts_data.sort_values(['Account', 'Date'])
            all_accounts_data.to_excel(writer, sheet_name='AccountData', index=False)
    
    print(f"Data successfully merged and written to {output_path}")

if __name__ == "__main__":
    main()

Starting Trader Tracking Data Merger...
Found 2 dashboard files and 2 performance files
Found 19 unique accounts in the data
Data successfully merged and written to C:\Users\Wolfrank\Desktop\TraderTracker\TraderTrackrr.xlsx


C:\Users\Wolfrank\AppData\Local\Temp\ipykernel_8472\2259058976.py:269: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_dfs, ignore_index=True)
